# 05 · Data Pipeline Orchestration

Build automated search → filter → download → export pipelines with PyGeoFetch YAML orchestration.

## Contents
1. Creating pipelines programmatically
2. YAML pipeline definition
3. Search, filter, download, export steps
4. Scheduling with cron
5. Running and monitoring
6. Pipeline history and logs

## 1 · Programmatic Pipeline Builder

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

# Build a pipeline step by step
pipeline = client.create_pipeline("weekly-london-sentinel2")

pipeline     .search(
        providers=["planetary_computer", "copernicus"],
        bbox=(-0.15, 51.47, -0.10, 51.52),
        date_range="last_7_days",
        cloud_cover="0-10",
        collections=["sentinel-2-l2a"],
    )     .filter(
        "data.cloud_cover < 8"
    )     .download(
        parallel=4,
        output="./pipelines/london/",
        post_process=["unzip", "reproject:EPSG:4326", "compress:lzw", "cog"],
        verify_checksum=True,
    )     .export(
        format="cloud_optimized_geotiff",
        destination="./archive/london/",
    )

print(pipeline)
print("Pipeline ready — call .run() to execute")

## 2 · YAML Pipeline Definition

Save pipelines as reusable YAML files — version-controlled and portable.

In [ ]:
PIPELINE_YAML = '''
name: weekly-sentinel2-london
description: Weekly Sentinel-2 acquisition for London with NDVI
schedule: "0 6 * * 1"          # Every Monday 06:00 UTC

steps:
  - search:
      providers:
        - planetary_computer
        - copernicus
        - aws_earth
      bbox: "-0.15,51.47,-0.10,51.52"
      date_range: last_7_days
      cloud_cover: 0-10
      collections:
        - sentinel-2-l2a
      max_results: 10

  - filter:
      expression: "data.cloud_cover < 8 AND data.resolution_m <= 10"
      max_items: 3

  - download:
      parallel: 4
      output: ./data/london/
      verify_checksum: true
      resume: true
      retry_attempts: 5
      post_process:
        - unzip
        - reproject:EPSG:4326
        - ndvi
        - compress:lzw
        - cog

  - export:
      format: cloud_optimized_geotiff
      destination: ./archive/london/
      filename_template: "{date}_{satellite}_{tile}_{cloud:.0f}pct.tif"
      overwrite: false
'''

# Save the YAML
from pathlib import Path
Path("./pipelines/london_weekly.yaml").parent.mkdir(exist_ok=True)
Path("./pipelines/london_weekly.yaml").write_text(PIPELINE_YAML)
print("Pipeline YAML saved to ./pipelines/london_weekly.yaml")
print()
print(PIPELINE_YAML)

## 3 · Running a Pipeline

In [ ]:
from pathlib import Path

# Validate before running
is_valid = client.data.validate_pipeline("./pipelines/london_weekly.yaml")
print(f"Pipeline valid: {is_valid}")

# Run the pipeline
result = client.run_pipeline_yaml("./pipelines/london_weekly.yaml")
print("\nPipeline result:")
import json
print(json.dumps(result, indent=2, default=str))

Pipeline valid: True

Pipeline result:
{
  "name": "weekly-sentinel2-london",
  "status": "completed",
  "steps_completed": 4,
  "scenes_found": 3,
  "scenes_downloaded": 3,
  "total_mb": 2142.5,
  "duration_seconds": 198.3,
  "output_files": [
    "./data/london/20240702_S2B_T30UXC_2pct.tif",
    "./data/london/20240625_S2B_T30UXC_3pct.tif",
    "./data/london/20240620_S2C_T30UXC_5pct.tif"
  ]
}


## 4 · Run Specific Steps Only

In [ ]:
# Run only the search step (useful for testing)
result = client.run_pipeline_yaml(
    "./pipelines/london_weekly.yaml",
    step="search",
)
print(f"Search step only: {result}")

## 5 · Scheduling Pipelines

In [ ]:
# Schedule for periodic execution
client.data.schedule_pipeline(
    "./pipelines/london_weekly.yaml",
    name="london-weekly",
    cron="0 6 * * 1",            # Every Monday 06:00 UTC
)

# List all scheduled pipelines
scheduled = client.data.list_scheduled_pipelines()
print("Scheduled pipelines:")
for p in scheduled:
    print(f"  {p['name']:<30} {p['cron']:<20} next: {p.get('next_run', 'N/A')}")

## 6 · Pipeline History and Logs

In [ ]:
# View pipeline execution history
history = client.data.pipeline_history(limit=5)
print(f"Last {len(history)} pipeline runs:")
for run in history:
    print(f"  {run['name']:<30} {run['status']:<10} {run['started_at']} → {run['duration_s']:.0f}s")

## 7 · End-to-End GeoAI Pipeline

In [ ]:
# PyGeoVision's end-to-end pipeline: data + AI in one call
result = client.pipeline(
    "building_footprints",
    bbox=(-0.15, 51.47, -0.10, 51.52),
    output_dir="./pipeline_output/london/",
    date="2024-06",
)

print(f"Pipeline result:")
print(f"  Status:      {'✓ Success' if result.success else '✗ Failed'}")
print(f"  Output:      {result.output_path}")
print(f"  Stats:       {result.stats}")
print(f"  Duration:    {result.duration_seconds:.1f}s")

## Available End-to-End Pipelines

In [ ]:
# List all 26 built-in pipelines
from pygeovision.ai.pipelines.domains import list_pipelines
pipes = list_pipelines()
print(f"Available pipelines ({len(pipes)}):")
for p in pipes:
    print(f"  client.pipeline('{p}', bbox=..., date=...)")

Available pipelines (26):
  client.pipeline('building_footprints', bbox=..., date=...)
  client.pipeline('canopy_height', bbox=..., date=...)
  client.pipeline('carbon_estimation', bbox=..., date=...)
  client.pipeline('change_detection', bbox=..., date=...)
  client.pipeline('coastal_monitoring', bbox=..., date=...)
  client.pipeline('crop_health', bbox=..., date=...)
  client.pipeline('crop_monitoring', bbox=..., date=...)
  client.pipeline('crop_type_mapping', bbox=..., date=...)
  client.pipeline('deforestation', bbox=..., date=...)
  client.pipeline('disaster_assessment', bbox=..., date=...)
  client.pipeline('flood_mapping', bbox=..., date=...)
  client.pipeline('forest_fire', bbox=..., date=...)
  client.pipeline('infrastructure_monitoring', bbox=..., date=...)
  client.pipeline('irrigation_detection', bbox=..., date=...)
  client.pipeline('land_cover', bbox=..., date=...)
  client.pipeline('land_surface_temperature', bbox=..., date=...)
  client.pipeline('landslide_detection', 